In [1]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:

!pip install pandas numpy scikit-learn scipy tqdm
!pip install torch torchvision torchaudio
!pip install transformers datasets
!pip install matplotlib seaborn
!pip install textattack
!pip install nltk

# --- Optional: If you get errors with textattack, install these specific versions ---
# !pip install tensorflow  # Only needed if you use specific Universal Sentence Encoder constraints
# !pip install editdistance

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [4]:
!pip install datasets pandas

Defaulting to user installation because normal site-packages is not writeable


In [5]:
%pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
%pip install transformers datasets scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
%pip install datasets==3.6.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd
import numpy as np
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
dataset = load_dataset("Hello-SimpleAI/HC3", name="all")

In [10]:
data_rows = []

for item in dataset['train']:
    # Add human answers (Label 0)
    for human_ans in item['human_answers']:
        data_rows.append({'text': human_ans, 'label': 0})
        
    # Add ChatGPT answers (Label 1)
    for chatgpt_ans in item['chatgpt_answers']:
        data_rows.append({'text': chatgpt_ans, 'label': 1})

In [11]:
df = pd.DataFrame(data_rows)

In [12]:
df.head()

,text,label
0,"Basically there are many categories of "" Best ...",0
1,"If you 're hearing about it , it 's because it...",0
2,"One reason is lots of catagories . However , h...",0
3,There are many different best seller lists tha...,1
4,salt is good for not dying in car crashes and ...,0


In [13]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

In [14]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [15]:
print("Loading Tokenizer...")
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading Tokenizer...


In [16]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

In [17]:
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 17090/17090 [00:02<00:00, 6392.40 examples/s]


In [18]:
print("Loading RoBERTa Model...")
# num_labels=2 because it's binary classification (Human vs AI)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading RoBERTa Model...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1073.49it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

In [19]:
import torch
print(f"Is CUDA (NVIDIA GPU) available? : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name : {torch.cuda.get_device_name(0)}")

Is CUDA (NVIDIA GPU) available? : True
GPU Name : NVIDIA RTX A4000


In [20]:
# Define the device explicitly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

# Move the RoBERTa model to the NVIDIA GPU
model = model.to(device)

Active Device: cuda


In [21]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [22]:
import torch
device = torch.device("cuda")
model = model.to(device)
print(f"Model successfully moved to: {device}")

Model successfully moved to: cuda


In [23]:
%pip install "accelerate>=1.1.0"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
# 7. Define Training Arguments (Hyperparameters + GPU Optimization)
training_args = TrainingArguments(
    output_dir="./roberta-hc3-detector",
    eval_strategy="epoch",       # <--- THIS IS THE FIX
    save_strategy="epoch",
    learning_rate=2e-5,          
    num_train_epochs=3,          
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir='./logs',
    
    # --- GPU & HARDWARE OPTIMIZATION SETTINGS ---
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16,
    fp16=True,                      
    dataloader_num_workers=4,       
    # --------------------------------------------
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

In [26]:
print("Starting Training...")
trainer.train()

Starting Training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.008498,0.005044,0.998947,0.998328,0.997957,0.998699
2,0.001829,0.017351,0.997894,0.996664,0.993901,0.999442
3,0.000006,0.015834,0.997718,0.996387,0.993351,0.999442


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.16it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

TrainOutput(global_step=12819, training_loss=0.007996558404924792, metrics={'train_runtime': 1744.8929, 'train_samples_per_second': 117.53, 'train_steps_per_second': 7.347, 'total_flos': 2.697901295003136e+16, 'train_loss': 0.007996558404924792, 'epoch': 3.0})

In [27]:
trainer.save_model("./final-roberta-hc3")
tokenizer.save_pretrained("./final-roberta-hc3")
print("Training Complete and Model Saved!")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.83it/s]

Training Complete and Model Saved!
